[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/plain_python.ipynb)

# Data-analysis assistant — plain Python

**Task.** Produce a supported household-waste summary without allowing model-generated code to execute.

**Read this notebook as:** choose a provider → load data → declare typed boundaries → run and validate the analysis.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "analysis-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import csv
import json
from pathlib import Path
from typing import ClassVar, Literal

from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture_path = ROOT / "data/data_analysis_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Visible workflow and safety boundary
The LLM never supplies executable code or numeric results. It chooses an enumerated tool and columns; `AnalysisRequest` rejects unsupported tools, columns and extra arguments. Empty data, schema mismatch or result mismatch stops with a deterministic fallback report.

In [ ]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class AnalysisDecision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: Literal["mean_reduction_by_intervention"]
    group_column: Literal["intervention"]
    before_column: Literal["before_kg"]
    after_column: Literal["after_kg"]


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


AnalysisDecision.model_rebuild(_types_namespace={"Literal": Literal})


def model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, prompt):
    response = await client.generate([user(prompt)], response_schema=schema)
    return schema.model_validate(response.structured_output)


async def run_analysis():
    client = model()
    trace = []
    with data_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    columns = tuple(rows[0]) if rows else ()
    trace.append({"event": "inspect", "rows": len(rows), "columns": columns})
    decision = await propose(
        client,
        AnalysisDecision,
        f"For columns {columns}, select tool mean_reduction_by_intervention with "
        "group_column intervention, before_column before_kg and after_column after_kg.",
    )
    if decision.tool != "mean_reduction_by_intervention":
        return {
            "outcome": "fallback",
            "trace": [*trace, {"event": "invalid_tool"}],
            "usage": {"model_calls": 1},
        }
    request = AnalysisRequest(
        group_column=decision.group_column,
        before_column=decision.before_column,
        after_column=decision.after_column,
    )
    trace.append({"event": "validated_request", "request": request.model_dump()})
    result = summarise_reduction(data_path, request)
    result_valid = result == expected["mean_reduction_kg"]
    trace.append({"event": "execute_and_validate", "result": result, "valid": result_valid})
    if not result_valid:
        return {"outcome": "fallback", "trace": trace, "usage": {"model_calls": 1, "tool_calls": 1}}
    report = await propose(
        client,
        Report,
        f"Report only these validated results: {result}. "
        "Include every result key once in groups and do not invent values.",
    )
    groups_valid = set(report.groups) == set(result)
    trace.append({"event": "report_validation", "valid": groups_valid})
    return {
        "outcome": "supported_report" if groups_valid else "fallback",
        "report": report.model_dump(),
        "code": "summarise_reduction(data_path, validated_request)",
        "outputs": result,
        "provenance": {
            "dataset_sha256": file_sha256(data_path),
            "fixture_version": fixture["fixture_version"],
        },
        "trace": trace,
        "termination": "criteria_met" if groups_valid else "result_validation_failed",
        "usage": {"model_calls": 2, "tool_calls": 1},
    }


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_analysis()
second = await run_analysis() if MODEL_PROVIDER == "mock" else first
try:
    AnalysisRequest(group_column="missing", before_column="before_kg", after_column="after_kg")
    schema_mismatch_rejected = False
except ValueError:
    schema_mismatch_rejected = True
failure_demonstrations = {
    "invalid_tool": "blocked before execution",
    "unsafe_code": "never evaluated",
    "schema_mismatch": schema_mismatch_rejected,
    "result_mismatch": "fallback before report",
}
evaluation = {
    "component": first["outputs"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["usage"]["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["code"].startswith("summarise_reduction") and schema_mismatch_rejected,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "result": first,
    "resource_report": first["usage"],
    "failure_demonstrations": failure_demonstrations,
    "fallback": "report validated deterministic results or stop",
}

## Limitations
This small descriptive dataset supports no causal inference. The allowlisted analysis intentionally excludes arbitrary model-authored code and advanced statistics.